In [3]:
import gpt as g
import numpy as np
#import matplotlib.pyplot as plt
#from matplotlib import gridspec
from itertools import permutations

#import scipy
#import math
#from scipy import special as sp
import time
import os
import glob

size = 8
grid = g.grid([size, size, size, size], g.double)
rng = g.random(str(time.time()))

U = g.qcd.gauge.unit(grid)
#W = g.qcd.gauge.unit(grid)
#rng.normal_element(U)

action_gauge = g.qcd.gauge.action.wilson(10.0)

metro = g.algorithms.markov.metropolis(rng)
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()
pure_gauge = True

# thermalize lattice first ##################################
beta = g.default.get_float("--beta", 10.0)
seed = g.default.get("--seed", "hmc-pure-gauge")
ntherm = g.default.get_int("--ntherm", 10)
n = g.default.get_int("--n", 200)
nwrite = g.default.get_int("--nwrite", 10)
g.default.set_verbose("omf4")

# conjugate momenta
mom = g.group.cartesian(U)
rng.normal_element(mom)

# Log
g.message(f"Lattice = {grid.fdimensions}")
g.message("Actions:")
# action for conj. momenta
a0 = g.qcd.scalar.action.mass_term()
g.message(f" - {a0.__name__}")

# wilson action 
a1 = g.qcd.gauge.action.wilson(beta)
g.message(f" - {a1.__name__}")


def hamiltonian():
    return a0(mom) + a1(U)

# molecular dynamics
sympl = g.algorithms.integrator.symplectic

iphmc = sympl.update_p(mom, lambda: a1.gradient(U, U))
iqhmc = sympl.update_q(U, lambda: a0.gradient(mom, mom))

# integrator
mdint_hmc = sympl.leap_frog(80, iqhmc, iphmc)
#g.message(f"Integration scheme:\n{mdint}")

# metropolis
metro = g.algorithms.markov.metropolis(rng)

# MD units
tau = 1.0
g.message(f"tau = {tau} MD units")


def hmc(tau, mom):
    rng.normal_element(mom)
    accrej = metro(U)
    h0 = hamiltonian()
    mdint_hmc(tau)
    h1 = hamiltonian()
    return [accrej(h1, h0), h1 - h0]

a = False
if a:
    b = 1
    
else:
    # thermalization
    for i in range(1, 10):
        h = []
        timer = g.timer("hmc")
        for _ in range(10):
            timer("trajectory")
            h += [hmc(tau, mom)]
        h = np.array(h)
        timer()
        g.message(f"{i*10} % of thermalization completed")
        g.message(timer)
        g.message(
            f"Plaquette = {g.qcd.gauge.plaquette(U)}, Acceptance = {np.mean(h[:,0]):.2f}, |dH| = {np.mean(np.abs(h[:,1])):.4e}"
        )


    start = time.time()

    # production

    history = []
    plaq = []

    for i in range(50):
        history += [hmc(tau, mom)]
        P = g.qcd.gauge.plaquette(U)
        plaq.append(P)
        g.message(f"Trajectory {i}, P={P}")

    end = time.time()
    print("time taken = ", end - start)

    history = np.array(history)
    g.message(f"Acceptance rate = {np.mean(history[:,0]):.2f}")
    g.message(f"<|dH|> = {np.mean(np.abs(history[:,1])):.4e}")
    ########################################################################################################################################


GPT :     163.504465 s : Initializing gpt.random(1757689834.64657,vectorized_ranlux24_389_64) took 0.000715017 s
GPT :     163.538108 s : Lattice = [8, 8, 8, 8]
GPT :     163.538629 s : Actions:
GPT :     163.538946 s :  - mass_term(m^-1 = 1.0)
GPT :     163.539331 s :  - wilson(10.0)
GPT :     163.540185 s : tau = 1.0 MD units
GPT :     173.683059 s : 10 % of thermalization completed
GPT :     173.683385 s : hmc:
                       : trajectory           1.01e+01 s (= 100.00 %); time/s = 9.88e-01/1.04e+00/1.01e+00 (min/max/avg)
GPT :     173.684114 s : Plaquette = 0.802001221756623, Acceptance = 1.00, |dH| = 2.2332e+00
GPT :     183.827447 s : 20 % of thermalization completed
GPT :     183.827787 s : hmc:
                       : trajectory           1.01e+01 s (= 100.00 %); time/s = 9.77e-01/1.03e+00/1.01e+00 (min/max/avg)
GPT :     183.828519 s : Plaquette = 0.7862782716797064, Acceptance = 1.00, |dH| = 1.1803e-01
GPT :     194.165113 s : 30 % of thermalization completed
GPT :  

Exception ignored in: <function lattice.__del__ at 0x10e4d0c20>
Traceback (most recent call last):
  File "/Users/ell579/Documents/Physics/lattice/gpt/lib/gpt/core/lattice.py", line 107, in __del__
    cgpt.delete_lattice(o)
KeyboardInterrupt: 


GPT :     214.927071 s : 50 % of thermalization completed
GPT :     214.927398 s : hmc:
                       : trajectory           1.04e+01 s (= 100.00 %); time/s = 1.00e+00/1.12e+00/1.04e+00 (min/max/avg)
GPT :     214.928177 s : Plaquette = 0.7818059763803932, Acceptance = 1.00, |dH| = 8.0182e-02
GPT :     225.167231 s : 60 % of thermalization completed
GPT :     225.167553 s : hmc:
                       : trajectory           1.02e+01 s (= 100.00 %); time/s = 9.90e-01/1.07e+00/1.02e+00 (min/max/avg)
GPT :     225.168189 s : Plaquette = 0.7828854373392541, Acceptance = 1.00, |dH| = 2.8603e-02
GPT :     235.524185 s : 70 % of thermalization completed
GPT :     235.524526 s : hmc:
                       : trajectory           1.04e+01 s (= 100.00 %); time/s = 9.80e-01/1.13e+00/1.04e+00 (min/max/avg)
GPT :     235.525274 s : Plaquette = 0.7842638970818631, Acceptance = 1.00, |dH| = 4.9312e-02
GPT :     245.723565 s : 80 % of thermalization completed
GPT :     245.723892 s : hmc:
   

In [ ]:
import gpt as g
import numpy as np
#import matplotlib.pyplot as plt
#from matplotlib import gridspec
from itertools import permutations

#import scipy
#import math
#from scipy import special as sp
import time
import os
import glob

size = 8
grid = g.grid([size, size, size, size], g.double)
rng = g.random(str(time.time()))

U = g.qcd.gauge.unit(grid)
W = g.qcd.gauge.unit(grid)
rng.normal_element(W)


action_gauge = g.qcd.gauge.action.wilson(10.0)


metro = g.algorithms.markov.metropolis(rng)
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()
pure_gauge = True

# thermalize lattice first ##################################
beta = g.default.get_float("--beta", 10.0)
seed = g.default.get("--seed", "hmc-pure-gauge")
ntherm = g.default.get_int("--ntherm", 10)
n = g.default.get_int("--n", 200)
nwrite = g.default.get_int("--nwrite", 10)
g.default.set_verbose("omf4")

# conjugate momenta
mom = g.group.cartesian(W)



In [12]:
iphmc = sympl.update_p(mom, lambda: a1.gradient(U, U))
iqhmc = sympl.update_q(U, lambda: a0.gradient(mom, mom))

# integrator
mdint_hmc_start = sympl.leap_frog(20, iphmc, iqhmc)

print(g.qcd.gauge.plaquette(U))

mdint_hmc_start(0.2)

print(g.qcd.gauge.plaquette(U))

1.0
0.9719455346976104
